# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmanBanik/Intern_at_fly/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

The raw grain from the warehouse is `report_date × client × content`. For our **Freestyle B** momentum prediction task, our contracted unit of analysis is **One row = One content item** (unique `content_hash_id`). We evaluate this at a fixed evaluation date, using a 90-day historical *feature window* and a 30-day future *target window*.

In [1]:
from dotenv import load_dotenv
import os
import duckdb

# Load the Hugging Face token safely using dotenv
load_dotenv('../../.env')
HF_TOKEN = os.environ.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
SAMPLE_TABLE = f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')"

print("DuckDB connected securely to HF!")

DuckDB connected securely to HF!


## 2. Fields: feature / label / context / excluded

*   **Context:** `client_hash_id`, `content_hash_id` (IDs used only for grouping, never for learning).
*   **Features:** Historical `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (aggregated over our feature window, e.g., the last 90 days before evaluation).
*   **Label:** A derived binary target (`future_traffic_dropped`) calculated purely from traffic in the future time window (e.g., >15% impression drop in the next 30 days).
*   **Excluded:** `ga4_sessions` (for rows where `ga4_data_available` is False). Also strictly excluding `health_score` and `action_type` to prevent circular target leakage, as they represent internal product decisions rather than pure observed signals.

## 3. Verify it with queries (grain, counts, missing values, windows)

In [2]:
# 1. Check grain: Confirm client_hash_id + content_hash_id + report_date is strictly unique
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) c
    FROM {SAMPLE_TABLE}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Duplicates found violating grain: {len(grain_check)}")

# 2. Check windows
windows = con.sql(f"""
    SELECT MIN(report_date) as start_date, MAX(report_date) as end_date
    FROM {SAMPLE_TABLE}
""").df()
print(f"Sample window spans from {windows['start_date'][0]} to {windows['end_date'][0]}")

# 3. Check GA4 missingness (the ga4 trap)
ga4_missing = con.sql(f"""
    SELECT 
        AVG(CASE WHEN ga4_data_available = FALSE THEN 1.0 ELSE 0.0 END) as pct_untracked
    FROM {SAMPLE_TABLE}
""").df()
print(f"Rows missing GA4 tracking (ga4_data_available=False): {ga4_missing['pct_untracked'][0]:.1%}")

Duplicates found violating grain: 5
Sample window spans from 2026-06-01 00:00:00 to 2026-06-30 00:00:00
Rows missing GA4 tracking (ga4_data_available=False): 74.0%


## 4. Data limits

*   **The Unbalanced Panel:** Clients onboarded at different times. We cannot assume every content item has a full 12+ months of history.
*   **The GA4 Trap:** We just verified that a massive portion (exactly 74.0%) of the sample rows are missing GA4 tracking (`ga4_data_available = False`). These rows have `0` sessions, but they mean "untracked", not "zero engagement". We must filter by the flag.
*   **No Causal Guarantees:** We are building a predictive momentum model based on historical correlation. The data cannot tell us *why* traffic dropped (e.g., algorithmic penalty vs seasonal shift), only that the warning signs were there.

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.